In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import seaborn as sns
import pandas as pd
import random
import hydra
import lightning as lit
import torch
import torch.nn as nn
import wandb
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import LearningRateMonitor, EarlyStopping, RichProgressBar, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger, CSVLogger
from loguru import logger
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf

from datetime import datetime

from machine_intelligence.models.SAnD import SAnD
from machine_intelligence.utilities.data_handling import load_parquet_table
from machine_intelligence.data.SepsisPredDataModule import SepsisPredDataModule
from machine_intelligence.data.TransformsData import ZeroFill

In [ ]:
cwd = Path.cwd()
cwd

In [ ]:
project_root = Path(cwd).parents[1]
project_root


In [ ]:
current_date = datetime.now().strftime('%Y-%m-%d')
plot_output_dir = project_root / 'plot_outputs' / current_date
plot_output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
model_path = project_root / Path("experiment_logs/sand-debug/version_192/checkpoints/epoch=19-step=620.ckpt")
model = SAnD.load_from_checkpoint(model_path)

In [ ]:
with initialize(version_base=None, config_path="../../configs"):
    cfg = compose(config_name="run_training.yaml", overrides=["data=cohort_a_r0.4_prev_0.3_imp_none"])

    print(cfg)

In [ ]:
wandb.init(mode="disabled")

In [ ]:
sepsis_pred_data = SepsisPredDataModule(cfg=cfg)

In [ ]:
sepsis_pred_data.setup(stage="test")

inference_dl = sepsis_pred_data.test_dataloader()

In [ ]:
sepsis_pred_data.considered_features

In [ ]:
batch = next(iter(inference_dl))

In [ ]:
batch_features_ts, batch_features_stat, batch_labels = batch

In [ ]:
sample_ts = batch_features_ts[1].unsqueeze(0)
sample_stat = batch_features_stat[1].unsqueeze(0)
sample_label = batch_labels[1]

sample_ts_2 = batch_features_ts[2].unsqueeze(0)
sample_stat_2 = batch_features_stat[2].unsqueeze(0)
sample_label_2 = batch_labels[2]



In [ ]:
sample_ts.shape

In [ ]:
print(model)

In [ ]:
model.encoder.blocks[0].attention.layer.out_proj

In [ ]:
attention = {}


def get_attention_hook(model, input, output):
    # model.attention_weights = output.detach()
    attention["attention"] = output.detach()

    # return hook

In [ ]:
# model.encoder.blocks[-1].attention_wo_res.register_forward_hook(get_attention_hook)
model.encoder.blocks[0].attention.layer.out_proj.register_forward_hook(get_attention_hook)


In [ ]:
model.to("cpu")
model.eval()

with torch.no_grad():
    output = model(sample_ts, sample_stat)

output

In [ ]:
print(model)

In [ ]:
attention_scores = model.encoder.blocks[0].attention.attn_weights.squeeze(0)
attention_scores.shape

In [ ]:

sns.heatmap(attention_scores.T)

In [ ]:
attention_weights = model.encoder.blocks[0].attention.attn_weights.squeeze(0)

In [ ]:
sns.heatmap(attention_weights)

In [ ]:

df_sample_ts = pd.DataFrame(sample_ts.squeeze(0), columns=sepsis_pred_data.considered_features)
zero_columns = df_sample_ts.loc[:, (df_sample_ts == 0).all()].columns.to_list()
df_sample_ts = df_sample_ts.drop(columns=zero_columns)
df_sample_ts.describe()

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(20, 15), sharex=True)
sns.lineplot(df_sample_ts, ax=axs[0])
sns.heatmap(attention_weights, ax=axs[1], cmap=sns.color_palette("icefire", as_cmap=True), cbar=False).legend([])
sns.heatmap(attention_scores.T, ax=axs[2], cmap=sns.color_palette("icefire", as_cmap=True), cbar=False).legend([])
plt.tight_layout()


In [ ]:
fig = plt.figure(figsize=(20, 15))
gs = GridSpec(3, 2, width_ratios=[20, 1], height_ratios=[1, 1, 1])

# sns.set(style="ticks")
# Plot the line plot
ax0 = fig.add_subplot(gs[0, 0])
sns.lineplot(df_sample_ts, ax=ax0)
ax0.margins(x=0)

# Plot the first heatmap
ax1 = fig.add_subplot(gs[1, 0])
sns.heatmap(attention_weights, ax=ax1, cbar=False)

# Plot the color bar for the first heatmap
cbar_ax1 = fig.add_subplot(gs[1, 1])
fig.colorbar(ax1.collections[0], cax=cbar_ax1)

# Plot the second heatmap
ax2 = fig.add_subplot(gs[2, 0])

sns.heatmap(attention_scores.T, ax=ax2, cmap=sns.color_palette("icefire", as_cmap=True), cbar=False)
# Plot the color bar for the second heatmap
cbar_ax2 = fig.add_subplot(gs[2, 1])
fig.colorbar(ax2.collections[0], cax=cbar_ax2)

# Adjust layout
plt.tight_layout()
plt.show()

Getting samples with a sepsis prediction



In [ ]:
batch_predictions = torch.nn.functional.sigmoid(model(batch_features_ts, batch_features_stat))


In [ ]:
batch_preds_thresholded = torch.threshold(batch_predictions, threshold=0.5, value=0).squeeze()
batch_preds_thresholded

In [ ]:
batch_predictions_sepsis = torch.nonzero(batch_preds_thresholded).squeeze()

below_threshold = batch_preds_thresholded < 0.5
below_threshold

In [ ]:
batch_predictions_no_sepsis = below_threshold.nonzero()

In [ ]:
from machine_intelligence.utilities.viz_setup import figure_setup_grid

from machine_intelligence.utilities.viz_setup import figure_setup, five_min_to_hours
import matplotlib.ticker as ticker


def plot_sand_attention(model, batch_ts, batch_stat, sample_idx):
    sample_ts = batch_ts[sample_idx].unsqueeze(0)
    sample_stat = batch_stat[sample_idx].unsqueeze(0)

    model.to("cpu")
    model.eval()

    with torch.no_grad():
        output = model(sample_ts, sample_stat)

    attention_weights = model.encoder.blocks[0].attention.attn_weights.squeeze(0)

    df_sample_ts = pd.DataFrame(sample_ts.squeeze(0), columns=sepsis_pred_data.considered_features)
    zero_columns = df_sample_ts.loc[:, (df_sample_ts == 0).all()].columns.to_list()
    df_sample_ts = df_sample_ts.drop(columns=zero_columns)

    fig, gs, axs, font_labels, font_annotations = figure_setup_grid(3, 2, width_cm=14, height_cm=18, num_colors=10,
                                                                    height_ratios=[1, 1, 1])

    # gs.update(left=0.05, right=0.95, wspace=0.05, hspace=0.05)
    # gs.update(left=0.05, right=0.95, wspace=0.05, hspace=)

    ax0 = axs[0, 0]
    ax1 = axs[1, 0]
    markers = ['o', 's', 'D', '^', 'v', '<', '>', 'p', 'h', '*', '+', 'x', 'X', 'd']
    # linestyles = ["-", "--", "-.", ":", "-:"]
    for col in df_sample_ts.columns:
        if "labname" in col:
            marker = random.choice(markers)
            sns.scatterplot(df_sample_ts[col], ax=ax1, label=col, marker=marker, s=50, zorder=100, alpha=0.95)
        else:

            # ls = random.choice(linestyles)
            # ls = linestyles
            # sns.lineplot(df_sample_ts[col], ax=ax0, label=col, ls=ls)
            sns.lineplot(df_sample_ts[col], ax=ax0, label=col)

    ax0.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.4),
        ncol=3,
        frameon=False,
        fancybox=True,
        shadow=True,
        prop=font_annotations,
    )
    ax1.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.4),
        ncol=3,
        frameon=False,
        fancybox=True,
        shadow=True,
        prop=font_annotations,
    )

    fig.delaxes(axs[0, 1])
    fig.delaxes(axs[1, 1])

    ax2 = axs[2, 0]
    sns.heatmap(attention_weights, ax=ax2, cbar=False)

    cbar_ax2 = axs[2, 1]
    fig.colorbar(ax1.collections[0], cax=cbar_ax2)

    five_min_hour_formatter = ticker.FuncFormatter(func=five_min_to_hours)
    ax0.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax0.xaxis.set_major_formatter(five_min_hour_formatter)

    ax1.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax1.xaxis.set_major_formatter(five_min_hour_formatter)

    ax2.xaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax2.xaxis.set_major_formatter(five_min_hour_formatter)

    ax2.yaxis.set_major_locator(plt.MultipleLocator(1 * 12))  # 5 min  x 12
    ax2.yaxis.set_major_formatter(five_min_hour_formatter)

    ax0.set_xlabel("Time [h]")
    ax1.set_xlabel("Time [h]")
    ax2.set_xlabel("Time [h]")
    ax2.set_ylabel("Time [h]")

    ax0.set_ylabel("Dense \n Time Series Variables",
                   )

    ax1.set_ylabel("Sparse \n Time Series Variables",
                   )

    cbar_ax2.set_ylabel("Attention Weight",
                        )

    ax0.margins(x=0)
    ax1.margins(x=0)

    # plt.subplots_adjust(hspace=0.2)

    # Adjust layout
    plt.tight_layout()
    plt.show()

    return fig, axs


In [ ]:
sand_attention_weights, _ = plot_sand_attention(model=model, batch_ts=batch_features_ts, batch_stat=batch_features_stat,
                                                sample_idx=13)

if save_as_svg := True:
    sand_attention_weights.savefig(plot_output_dir / "sand_attention_weights.svg", format="svg")

In [ ]:
print(model)

In [ ]:
model.to("cpu")
model.eval()

with torch.no_grad():
    output = torch.nn.functional.sigmoid(model(sample_ts_2, sample_stat_2))

output


In [ ]:

attention_scores = model.encoder.blocks[0].attention.attn_weights.squeeze(0)

sns.heatmap(attention_scores.T)

In [ ]:
attention_weights = model.encoder.blocks[0].attention.attn_weights.squeeze(0)

In [ ]:
sns.heatmap(attention_weights)


 model.encoder.blocks[0].attention.attn_weights.squeeze(0)

model_attention = attention["attention"]

model_attention = model_attention.squeeze(0)

import seaborn as sns
import matplotlib.pyplot as plt



attention_change = torch.gradient(model_attention)[0]

1) Time Series Data of the Samples
2) Raw Attention 
3) Rate of Attention change (derivative)

fig, axs = plt.subplots(3, 1, figsize=(20, 15))

sns.lineplot(sample_ts.squeeze(0), ax=axs[0]).legend([])
sns.lineplot(model_attention, ax=axs[1]).legend([])
sns.lineplot(attention_change, ax=axs[2]).legend([])

import scipy.integrate as integrate

attention_integrated = integrate.cumulative_trapezoid(attention_change)

sns.lineplot(attention_integrated).legend([])

attention_integrated = integrate.cumulative_trapezoid(model_attention)
sns.lineplot(attention_integrated).legend([])


attention_change_abs = abs(attention_change)

attention_integrated = integrate.cumulative_trapezoid(attention_change_abs)
sns.lineplot(attention_integrated).legend([])


fig, axs = plt.subplots(3, 1, figsize=(20, 15))

sns.lineplot(sample_ts.squeeze(0), ax=axs[0]).legend([])
sns.lineplot(attention_change, ax=axs[1]).legend([])
sns.lineplot(attention_integrated).legend([])



attention - embedding bzw. - skip connection input ?